# Text Generation with GPT-2 — Reference Notebook

> **Reference notebook** — kept up to date with current `transformers` usage. See
> [`transformers_architectures.md`](../03-transformers-and-llms-p1/transformers_architectures.md#gpt-generative-pre-trained-transformer--decoder-only)
> for the underlying GPT/decoder-only concept, and [`README.md`](./README.md) for the fuller
> walkthrough this notebook distills.

**Methods covered:**
- Loading a pretrained decoder-only causal LM (`GPT2LMHeadModel`) and its matching tokenizer (`GPT2Tokenizer`)
- Encoding a prompt into token ids plus an explicit attention mask
- Autoregressive generation via `.generate()` using **beam search** (`num_beams`, `no_repeat_ngram_size`, `early_stopping`)
- Decoding generated token ids back into text

**Use this as a reference when:** you need copy-paste-ready code for loading a Hugging Face causal LM
and generating text with beam search.

**Don't use this as a reference for:** sampling-based decoding (top-k / top-p / temperature — not shown
here), fine-tuning, or chat-style/multi-turn generation — none of that is covered in this notebook.

In [ ]:
# Only transformers + torch are needed here -- GPT-2 generation runs entirely on the
# PyTorch backend (return_tensors="pt"), so no TensorFlow install/import is required.
!pip install -q transformers torch

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

In [ ]:
# GPT2Tokenizer converts text <-> token ids using the exact vocabulary GPT-2 was pretrained
# with; it must always come from the same checkpoint as the model.
tokenizer = GPT2Tokenizer.from_pretrained("gpt2-large")

In [ ]:
# GPT-2 ships with no dedicated padding token, so the end-of-sequence token id is reused as
# the pad token id -- required even for single-prompt generation, since `.generate()` needs
# a defined pad id internally.
model = GPT2LMHeadModel.from_pretrained("gpt2-large", pad_token_id=tokenizer.eos_token_id)

In [ ]:
prompt = "What is Quantum Mechanics?"

# Encode the prompt into token ids and build the matching attention mask explicitly --
# passing it avoids relying on the model to infer it (which it can't here, since the pad
# and eos token ids are identical).
encoded = tokenizer(prompt, return_tensors="pt")
input_ids = encoded["input_ids"]
attention_mask = encoded["attention_mask"]

In [ ]:
# Beam search: explore `num_beams` candidate continuations in parallel instead of always
# picking the single most likely next token, trading compute for more globally coherent text.
output_ids = model.generate(
    input_ids,
    attention_mask=attention_mask,
    max_length=100,          # total length, prompt included
    num_beams=5,              # candidate sequences tracked at each step
    no_repeat_ngram_size=2,   # forbid repeated 2-grams, curbing repetition loops
    early_stopping=True,      # stop once every beam has produced an end-of-sequence token
)

In [ ]:
# Convert generated token ids back to text; skip_special_tokens drops control tokens
# (like the reused pad/eos token) that aren't meant to appear in the final output.
generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print(generated_text)